# 🇳🇵 Hybrid Rule-Guided Subword Tokenizer for Nepali

**A research-grade evaluation notebook for the npltk hybrid tokenizer pipeline.**

This notebook implements:
1. **Stage 1** — Rule-based preprocessing (normalization, script detection, pre-tokenization)
2. **Stage 2** — SentencePiece Unigram subword model (8k, 16k, 32k vocab)
3. **Baseline tokenizers** — Whitespace, Regex, Raw SentencePiece
4. **Evaluation metrics** — OOV rate, tokens/sentence, compression ratio, vocab utilization
5. **Visualizations** — Bar graphs comparing all tokenizers

---

## 1. Setup

In [ ]:
!pip install sentencepiece==0.2.0 pandas matplotlib --quiet
print('✅ All libraries installed')

In [ ]:
import os
import re
import unicodedata
import random
import time
from collections import Counter
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Any

import sentencepiece as spm
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
random.seed(42)

# Output directory
OUTPUT_DIR = '/content/npltk_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Imports done')

## 2. Load Dataset

**Option A:** Mount Google Drive and load your real corpus.  
**Option B:** Use the embedded sample data below for quick testing.

Uncomment whichever option you need.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION A — Load from Google Drive (UNCOMMENT FOR REAL CORPUS)
# ══════════════════════════════════════════════════════════════════════════════
# from google.colab import drive
# drive.mount('/content/drive')
# CORPUS_PATH = '/content/drive/MyDrive/nepali_corpus.txt'   # ← CHANGE THIS
# with open(CORPUS_PATH, encoding='utf-8', errors='ignore') as f:
#     corpus_lines = [l.strip() for l in f if l.strip()]

# ══════════════════════════════════════════════════════════════════════════════
# OPTION B — Embedded sample data (for quick testing / demo)
# ══════════════════════════════════════════════════════════════════════════════
SAMPLE_CORPUS = [
    "नेपाल एक सुन्दर देश हो।",
    "काठमाडौं नेपालको राजधानी हो।",
    "घरहरूमा धेरै मान्छे बस्छन्।",
    "उहाँले भन्नुभयो कि यो गलत छ।",
    "२०२६ सालमा नेपालमा धेरै परिवर्तन भयो।",
    "खान्छन् भन्दा school राम्रो छ।",
    "Facebookमा धेरै मान्छे छन्।",
    "हामी सबै नेपाली हौं।",
    "नेपालबाट विदेश जाने मान्छे धेरै छन्।",
    "मेरो नाम राम हो र म काठमाडौंमा बस्छु।",
    "यो किताब राम्रो छ।",
    "तपाईंको नाम के हो?",
    "मलाई खाना खान मन लाग्छ।",
    "आज मौसम राम्रो छ।",
    "हिजो school बन्द थियो।",
    "उनीहरूले काम गरिरहेका छन्।",
    "मैले पुस्तक पढें।",
    "तिमीलाई कति पैसा चाहियो?",
    "नेपालको जनसंख्या लगभग तीन करोड छ।",
    "हामीले मिलेर काम गर्नुपर्छ।",
    "विद्यार्थीहरूले परीक्षा दिए।",
    "यो गाडी महंगो छ।",
    "बिहान सबेरै उठ्नु राम्रो हो।",
    "कृषि नेपालको प्रमुख व्यवसाय हो।",
    "हिमालय नेपालको गौरव हो।",
    "गरिबी हटाउनु सबैको कर्तव्य हो।",
    "सरकारले नयाँ नीति ल्याउनुपर्छ।",
    "शिक्षा सबैको अधिकार हो।",
    "प्रविधिको विकासले जीवन सजिलो बनाएको छ।",
    "स्वास्थ्य नै सबैभन्दा ठूलो धन हो।",
    "लोकतन्त्रमा जनताको आवाज सुनिनुपर्छ।",
    "पर्यावरण संरक्षण अत्यन्त जरुरी छ।",
    "भ्रष्टाचार देशको विकासमा बाधक हो।",
    "खेलकुदले राष्ट्रिय एकता बढाउँछ।",
    "संस्कृति र परम्पराको जगेर्ना गर्नुपर्छ।",
    "इन्टरनेटले संसारलाई जोडेको छ।",
    "महिला सशक्तिकरण आवश्यक छ।",
    "बाल अधिकारको रक्षा गर्नुपर्छ।",
    "दूध, दही र घिउ स्वास्थ्यका लागि राम्रो हो।",
    "नेपालमा पर्यटन उद्योग बढ्दै छ।",
    "haha kasto ramro din!! 🙂😊",
    "malai yo movie mann paryo bro 😍",
    "#NepaliPride trending chha twitter ma",
    "Instagramमा photo हाल्नु भो?",
    "COVID-19 ले गर्दा lockdown भयो।",
    "OMG!! kasto dherai traffic aaja 🚗🚗🚗",
    "बिहानको ६ बजे उठ्नुपर्छ।",
    "Rs. 500 मा खाना पाइन्छ।",
    "डा. शर्माले भन्नुभयो।",
    "श्री. राम बहादुर आउनुभयो।",
]


# Expand sample data by repeating with slight variations for training
corpus_lines = SAMPLE_CORPUS * 200  # 10,000 lines for demo
random.shuffle(corpus_lines)

print(f'Corpus size: {len(corpus_lines):,} lines')
print(f'Sample: {corpus_lines[0]}')

In [ ]:
# Save corpus to file (needed by SentencePiece trainer)
RAW_CORPUS_FILE = os.path.join(OUTPUT_DIR, 'raw_corpus.txt')
with open(RAW_CORPUS_FILE, 'w', encoding='utf-8') as f:
    for line in corpus_lines:
        f.write(line + '\n')

print(f'✅ Raw corpus saved: {RAW_CORPUS_FILE}')

## 3. Stage 1 — Rule-Based Preprocessing Pipeline

Seven normalization rules applied sequentially, each recording its transformation.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                      NORMALIZATION RULES                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

@dataclass
class Transform:
    """Records one normalization transformation."""
    rule: str
    before: str
    after: str
    meta: Dict[str, Any] = field(default_factory=dict)


# ── Space / invisible char maps ──────────────────────────────────────────────
SPACE_MAP = {
    '\u00A0': ' ',   # non-breaking space
    '\u202F': ' ',   # narrow non-breaking space
    '\u2009': ' ',   # thin space
    '\u200B': '',    # zero-width space
    '\uFEFF': '',    # byte-order mark
}

# Nepali digit → ASCII digit map
NEPALI_DIGIT_MAP = str.maketrans('०१२३४५६७८९', '0123456789')


# ── Rule 1: Unicode NFC normalization ────────────────────────────────────────
def rule_unicode_nfc(text: str) -> Tuple[str, Dict]:
    """Apply Unicode NFC normalization."""
    out = unicodedata.normalize('NFC', text)
    return out, {'changed': out != text}


# ── Rule 2: Whitespace normalization ─────────────────────────────────────────
def rule_whitespace(text: str) -> Tuple[str, Dict]:
    """Normalize invisible/exotic spaces and collapse multiple spaces."""
    before = text
    for k, v in SPACE_MAP.items():
        text = text.replace(k, v)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip(), {'changed': before != text}


# ── Rule 3: Remove invisible control characters ─────────────────────────────
def rule_remove_invisible(text: str) -> Tuple[str, Dict]:
    """Remove control characters (except newline and tab)."""
    removed = 0
    out = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat == 'Cc' and ch not in ('\n', '\t'):
            removed += 1
            continue
        out.append(ch)
    result = ''.join(out)
    return result, {'removed': removed}


# ── Rule 4: ZWJ / ZWNJ cleanup ──────────────────────────────────────────────
def rule_zwj_cleanup(text: str) -> Tuple[str, Dict]:
    """Remove zero-width joiner and non-joiner characters."""
    out = text.replace('\u200D', '').replace('\u200C', '')
    return out, {'changed': out != text}


# ── Rule 5: Halant cleanup ───────────────────────────────────────────────────
def rule_halant_cleanup(text: str) -> Tuple[str, Dict]:
    """Fix halant (virama) followed by spaces or repeated halants."""
    before = text
    text = re.sub(r'\u094D\s+', '\u094D', text)  # halant + spaces
    text = re.sub(r'\u094D{2,}', '\u094D', text)  # repeated halant
    return text, {'changed': before != text}


# ── Rule 6: Diacritic deduplication ──────────────────────────────────────────
def rule_diacritic_dedupe(text: str) -> Tuple[str, Dict]:
    """Remove repeated Anunasika (anusvara/chandrabindu)."""
    before = text
    text = re.sub(r'\u0902{2,}', '\u0902', text)  # repeated anusvara
    text = re.sub(r'\u0901{2,}', '\u0901', text)  # repeated chandrabindu
    return text, {'changed': before != text}


# ── Rule 7: Repetition compression (social media) ────────────────────────────
def rule_repetition_compress(text: str) -> Tuple[str, Dict]:
    """Compress characters repeated > 3 times (e.g. हाहाहाहा → हाहा)."""
    before = text
    # Compress single char repetitions > 3
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)
    return text, {'compressed': before != text}


# ── Full pipeline ────────────────────────────────────────────────────────────
RULES = [
    ('unicode_nfc', rule_unicode_nfc),
    ('whitespace', rule_whitespace),
    ('remove_invisible', rule_remove_invisible),
    ('zwj_cleanup', rule_zwj_cleanup),
    ('halant_cleanup', rule_halant_cleanup),
    ('diacritic_dedupe', rule_diacritic_dedupe),
    ('repetition_compress', rule_repetition_compress),
]


def normalize(text: str) -> Tuple[str, List[Transform]]:
    """Apply all normalization rules sequentially."""
    transforms = []
    current = text
    for name, fn in RULES:
        updated, meta = fn(current)
        if updated != current:
            transforms.append(Transform(rule=name, before=current, after=updated, meta=meta))
        current = updated
    return current, transforms


# ── Quick test ───────────────────────────────────────────────────────────────
test = '  घरमा  \u200B नेपाल\u200Dबाट   गएँ '
result, xforms = normalize(test)
print(f'IN : {repr(test)}')
print(f'OUT: {repr(result)}')
for t in xforms:
    print(f'  [{t.rule}] {t.meta}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                     PRE-TOKENIZER (Script-Aware)                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

DEV_RANGE  = r'\u0900-\u0963\u0966-\u097F'  # Devanagari block
NEP_DIGIT  = r'\u0966-\u096F'               # Nepali digits ०-९
PUNCT_CHARS = r'।!?.,;:…—–\-(){}\[\]<>«»\"\"\'\'\'/\\|@#%^&*_+=~`'


def _build_pattern():
    url    = r'(?P<URL>(https?://|www\.)\S+)'
    number = rf'(?P<NUMBER>[\d{NEP_DIGIT}]+(?:[,\.:/\-][\d{NEP_DIGIT}]+)*)'
    dev    = rf'(?P<DEVWORD>[{DEV_RANGE}]+)'
    lat    = r"(?P<LATWORD>[A-Za-z]+(?:'[A-Za-z]+)?)"
    punct  = rf'(?P<PUNCT>[{re.escape(PUNCT_CHARS)}])'
    ws     = r'(?P<WS>\s+)'
    other  = r'(?P<OTHER>.)'
    return re.compile('|'.join([url, number, dev, lat, punct, ws, other]))


MASTER_RE = _build_pattern()


@dataclass(frozen=True)
class PreToken:
    text: str
    start: int
    end: int
    type: str   # DEVWORD, LATWORD, NUMBER, PUNCT, URL, EMOJI, SYMBOL


def _is_emoji(ch: str) -> bool:
    code = ord(ch)
    return (0x1F300 <= code <= 0x1FAFF
            or 0x2600 <= code <= 0x26FF
            or 0x2700 <= code <= 0x27BF)


def pre_tokenize(text: str, keep_punct: bool = True) -> List[PreToken]:
    """Split normalized text into typed pre-token chunks."""
    tokens = []
    for m in MASTER_RE.finditer(text):
        kind = m.lastgroup
        val  = m.group(0)
        s, e = m.start(), m.end()

        if kind == 'WS':
            continue
        elif kind == 'URL':
            tokens.append(PreToken(val, s, e, 'URL'))
        elif kind == 'NUMBER':
            tokens.append(PreToken(val, s, e, 'NUMBER'))
        elif kind == 'DEVWORD':
            tokens.append(PreToken(val, s, e, 'DEVWORD'))
        elif kind == 'LATWORD':
            tokens.append(PreToken(val, s, e, 'LATWORD'))
        elif kind == 'PUNCT':
            if keep_punct:
                tokens.append(PreToken(val, s, e, 'PUNCT'))
        else:
            tp = 'EMOJI' if (len(val) == 1 and _is_emoji(val)) else 'SYMBOL'
            tokens.append(PreToken(val, s, e, tp))
    return tokens


# ── Test pre-tokenizer ───────────────────────────────────────────────────────
demo = 'खान्छन् भन्दा school राम्रो छ। मिति 2026/01/31 हो 🙂'
pts = pre_tokenize(demo)
print(f'Input: {demo}')
print(f'{"Text":<18} {"Type":<10} Span')
print('─' * 40)
for p in pts:
    print(f'{p.text:<18} {p.type:<10} ({p.start},{p.end})')

In [ ]:
# ── Apply Stage 1 to entire corpus ──────────────────────────────────────────
normalized_lines = []
for line in corpus_lines:
    normed, _ = normalize(line)
    if normed:
        normalized_lines.append(normed)

CLEAN_CORPUS_FILE = os.path.join(OUTPUT_DIR, 'clean_corpus.txt')
with open(CLEAN_CORPUS_FILE, 'w', encoding='utf-8') as f:
    for line in normalized_lines:
        f.write(line + '\n')

print(f'✅ Normalized corpus: {len(normalized_lines):,} lines → {CLEAN_CORPUS_FILE}')

## 4. Train SentencePiece Models (8k, 16k, 32k)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║               TRAIN SENTENCEPIECE UNIGRAM MODELS                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

VOCAB_SIZES = [8000, 16000, 32000]


def train_sp_model(input_file: str, prefix: str, vocab_size: int,
                   model_type: str = 'unigram') -> str:
    """Train a SentencePiece model and return path to .model file."""
    print(f'  Training {model_type} vocab_size={vocab_size:,}...')
    t0 = time.time()

    spm.SentencePieceTrainer.train(
        input=input_file,
        model_prefix=prefix,
        model_type=model_type,
        vocab_size=vocab_size,
        character_coverage=0.9995,
        input_sentence_size=min(2_000_000, len(normalized_lines)),
        shuffle_input_sentence=True,
        num_threads=4,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        user_defined_symbols=['।', '?', '!', ',', ';', ':', '…', '—', '–'],
        byte_fallback=True,
        normalization_rule_name='identity',  # our Stage 1 already normalizes
        split_by_unicode_script=True,
        split_by_whitespace=True,
    )

    elapsed = time.time() - t0
    model_path = f'{prefix}.model'
    size_kb = os.path.getsize(model_path) / 1024
    print(f'  ✅ Done in {elapsed:.1f}s  |  Model: {size_kb:.1f} KB')
    return model_path


# ── Train on CLEANED corpus (Stage 1 output) ────────────────────────────────
print('Training on normalized (Stage 1) corpus:\n')
hybrid_models = {}
for vs in VOCAB_SIZES:
    prefix = os.path.join(OUTPUT_DIR, f'hybrid_{vs}')
    path = train_sp_model(CLEAN_CORPUS_FILE, prefix, vs)
    hybrid_models[vs] = path

# ── Train on RAW corpus (for baseline comparison) ────────────────────────────
print('\nTraining on raw corpus (baseline):\n')
raw_sp_models = {}
for vs in VOCAB_SIZES:
    prefix = os.path.join(OUTPUT_DIR, f'raw_{vs}')
    path = train_sp_model(RAW_CORPUS_FILE, prefix, vs)
    raw_sp_models[vs] = path

print('\n✅ All models trained')

## 5. Tokenizer Integration — Hybrid Pipeline

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                    HYBRID TOKENIZER CLASS                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class HybridTokenizer:
    """Stage 1 (rules) + Stage 2 (SentencePiece) hybrid tokenizer."""

    def __init__(self, model_path: str, name: str = 'hybrid'):
        self.name = name
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(model_path)
        self.vocab_size = self.sp.get_piece_size()

    def tokenize(self, text: str) -> List[str]:
        """Full hybrid pipeline: normalize → pre-tokenize → subword."""
        # Stage 1: Normalize
        normed, _ = normalize(text)
        # Stage 1: Pre-tokenize
        pre_tokens = pre_tokenize(normed)

        tokens = []
        for pt in pre_tokens:
            if pt.type == 'DEVWORD':
                # Stage 2: Send Devanagari chunks through SP model
                pieces = self.sp.encode(pt.text, out_type=str)
                tokens.extend(pieces)
            else:
                # Non-Devanagari: pass through as-is from rule engine
                tokens.append(pt.text)
        return tokens

    def has_unk(self, text: str) -> int:
        """Count UNK tokens in the tokenized output."""
        normed, _ = normalize(text)
        pre_tokens = pre_tokenize(normed)
        unk_count = 0
        for pt in pre_tokens:
            if pt.type == 'DEVWORD':
                ids = self.sp.encode(pt.text, out_type=int)
                unk_count += ids.count(1)  # unk_id = 1
        return unk_count


# ── Quick test ───────────────────────────────────────────────────────────────
ht = HybridTokenizer(hybrid_models[16000], name='hybrid_16k')
demo_text = 'खान्छन् भन्दा school राम्रो छ। 🙂'
print(f'Input : {demo_text}')
print(f'Tokens: {ht.tokenize(demo_text)}')

## 6. Baseline Tokenizers

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                    BASELINE TOKENIZERS                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class WhitespaceTokenizer:
    """Baseline: naive split on whitespace."""
    name = 'whitespace'
    vocab_size = None  # unbounded

    def tokenize(self, text: str) -> List[str]:
        return text.strip().split()

    def has_unk(self, text: str) -> int:
        return 0  # whitespace tokenizer has no UNK concept


class RegexTokenizer:
    """Baseline: rule-based pre-tokenizer only (no subword model)."""
    name = 'regex'
    vocab_size = None

    def tokenize(self, text: str) -> List[str]:
        normed, _ = normalize(text)
        pts = pre_tokenize(normed)
        return [p.text for p in pts]

    def has_unk(self, text: str) -> int:
        return 0


class RawSPTokenizer:
    """Baseline: raw SentencePiece with NO preprocessing."""

    def __init__(self, model_path: str, name: str = 'raw_sp'):
        self.name = name
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(model_path)
        self.vocab_size = self.sp.get_piece_size()

    def tokenize(self, text: str) -> List[str]:
        return self.sp.encode(text, out_type=str)

    def has_unk(self, text: str) -> int:
        ids = self.sp.encode(text, out_type=int)
        return ids.count(1)


# ── Instantiate all tokenizers ───────────────────────────────────────────────
ALL_TOKENIZERS = [
    WhitespaceTokenizer(),
    RegexTokenizer(),
    RawSPTokenizer(raw_sp_models[16000], name='raw_sp_16k'),
    HybridTokenizer(hybrid_models[8000],  name='hybrid_8k'),
    HybridTokenizer(hybrid_models[16000], name='hybrid_16k'),
    HybridTokenizer(hybrid_models[32000], name='hybrid_32k'),
]

print('Tokenizers ready:')
for t in ALL_TOKENIZERS:
    print(f'  • {t.name}')

## 7. Evaluation Metrics

| Metric | Formula | Good value |
|---|---|---|
| OOV Rate | `unk_tokens / total_tokens` | Lower is better |
| Avg Token Length | `sum(len(tok)) / total_tokens` | Higher = fewer fragments |
| Tokens per Sentence | `total_tokens / total_sentences` | Lower = more efficient |
| Compression Ratio | `total_chars / total_tokens` | Higher = more compact |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                    EVALUATION METRICS                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Use a consistent evaluation set (deduplicated)
eval_set = list(set(corpus_lines))  # unique sentences
print(f'Evaluation set: {len(eval_set)} unique sentences')


def evaluate_tokenizer(tokenizer, sentences: List[str]) -> Dict[str, float]:
    """Compute evaluation metrics for one tokenizer."""
    total_tokens    = 0
    total_chars     = 0
    total_unk       = 0
    total_tok_len   = 0
    unique_tokens   = set()

    for sent in sentences:
        tokens = tokenizer.tokenize(sent)
        total_tokens  += len(tokens)
        total_chars   += len(sent)
        total_unk     += tokenizer.has_unk(sent)
        total_tok_len += sum(len(t) for t in tokens)
        unique_tokens.update(tokens)

    n_sents = len(sentences)

    oov_rate         = total_unk / max(total_tokens, 1)
    avg_tok_len      = total_tok_len / max(total_tokens, 1)
    tokens_per_sent  = total_tokens / max(n_sents, 1)
    compression_ratio = total_chars / max(total_tokens, 1)
    vocab_used       = len(unique_tokens)

    return {
        'tokenizer':        tokenizer.name,
        'oov_rate':         round(oov_rate, 5),
        'avg_token_length': round(avg_tok_len, 2),
        'tokens_per_sent':  round(tokens_per_sent, 2),
        'compression_ratio': round(compression_ratio, 2),
        'unique_tokens':    vocab_used,
        'total_tokens':     total_tokens,
    }


# ── Run evaluation on all tokenizers ─────────────────────────────────────────
results = []
for tok in ALL_TOKENIZERS:
    print(f'Evaluating: {tok.name}...')
    metrics = evaluate_tokenizer(tok, eval_set)
    results.append(metrics)

# Create DataFrame
df_results = pd.DataFrame(results)
df_results.set_index('tokenizer', inplace=True)

print('\n' + '═' * 80)
print('EVALUATION RESULTS')
print('═' * 80)
print(df_results.to_string())

## 8. 📊 Visualizations — Bar Graph Comparisons

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                    PLOTTING HELPER                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def plot_bar(df: pd.DataFrame, column: str, title: str, ylabel: str,
             filename: str = None, higher_is_better: bool = False):
    """
    Create a clean bar graph for one metric.
    Uses matplotlib default color cycle (no manual colors).
    """
    fig, ax = plt.subplots(figsize=(10, 5))

    tokenizers = df.index.tolist()
    values = df[column].tolist()

    bars = ax.bar(tokenizers, values)

    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f'{val}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Tokenizer', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    ax.set_axisbelow(True)

    # Rotate x-labels if needed
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()

    if filename:
        path = os.path.join(OUTPUT_DIR, filename)
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f'  Saved: {path}')

    plt.show()

print('✅ Plotting helper ready')

In [ ]:
# ── Graph 1: OOV Rate Comparison ────────────────────────────────────────────
plot_bar(
    df_results, 'oov_rate',
    title='OOV Rate Comparison',
    ylabel='OOV Rate (lower is better)',
    filename='graph_oov_rate.png'
)

In [ ]:
# ── Graph 2: Average Tokens per Sentence ────────────────────────────────────
plot_bar(
    df_results, 'tokens_per_sent',
    title='Average Tokens per Sentence',
    ylabel='Tokens / Sentence (lower = more efficient)',
    filename='graph_tokens_per_sent.png'
)

In [ ]:
# ── Graph 3: Compression Ratio ──────────────────────────────────────────────
plot_bar(
    df_results, 'compression_ratio',
    title='Compression Ratio',
    ylabel='Chars / Token (higher = more compact)',
    filename='graph_compression_ratio.png',
    higher_is_better=True
)

In [ ]:
# ── Graph 4: Vocabulary Usage (Unique Tokens Used) ──────────────────────────
plot_bar(
    df_results, 'unique_tokens',
    title='Vocabulary Utilization (Unique Tokens Used)',
    ylabel='Number of Unique Tokens',
    filename='graph_vocab_usage.png'
)

In [ ]:
# ── Graph 5: Average Token Length ───────────────────────────────────────────
plot_bar(
    df_results, 'avg_token_length',
    title='Average Token Length (characters)',
    ylabel='Avg Token Length (higher = less fragmented)',
    filename='graph_avg_token_length.png',
    higher_is_better=True
)

## 9. Example Outputs

Tokenization results on three text categories: formal, noisy, and mixed-language.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                    EXAMPLE TOKENIZATION OUTPUTS                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

EXAMPLE_TEXTS = {
    'FORMAL': [
        'नेपालको संविधान २०७२ सालमा जारी भयो।',
        'लोकतन्त्रमा जनताको आवाज सुनिनुपर्छ।',
        'प्रविधिको विकासले जीवन सजिलो बनाएको छ।',
    ],
    'NOISY / SOCIAL MEDIA': [
        'haha kasto ramro din!! 🙂😊',
        'OMG!! kasto dherai traffic aaja 🚗🚗🚗',
        'malai yo movie mann paryo bro 😍',
    ],
    'MIXED LANGUAGE': [
        'Facebookमा photo हाल्नु भो?',
        '#NepaliPride trending chha twitter ma',
        'COVID-19 ले गर्दा lockdown भयो।',
    ],
}

# Use hybrid_16k for demonstration
demo_tok = HybridTokenizer(hybrid_models[16000], name='hybrid_16k')

for category, texts in EXAMPLE_TEXTS.items():
    print(f'\n{"═"*70}')
    print(f'  {category}')
    print(f'{"═"*70}')
    for text in texts:
        tokens = demo_tok.tokenize(text)
        print(f'\n  Input : {text}')
        print(f'  Tokens: {tokens}')
        print(f'  Count : {len(tokens)} tokens')

In [ ]:
# ── Side-by-side comparison of ALL tokenizers on one sentence ────────────────
comparison_text = 'नेपालबाट विदेश जाने मान्छे धेरै छन्।'

print(f'Input: {comparison_text}\n')
print(f'{"Tokenizer":<18} {"Count":<8} Tokens')
print('─' * 70)

for tok in ALL_TOKENIZERS:
    tokens = tok.tokenize(comparison_text)
    print(f'{tok.name:<18} {len(tokens):<8} {tokens}')

## 10. Save All Outputs

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                    SAVE EVALUATION RESULTS                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# 1. Save evaluation metrics as CSV
csv_path = os.path.join(OUTPUT_DIR, 'evaluation_results.csv')
df_results.to_csv(csv_path)
print(f'✅ Evaluation CSV: {csv_path}')

# 2. List all saved models
print('\n✅ Trained models:')
for vs in VOCAB_SIZES:
    hybrid_path = hybrid_models[vs]
    raw_path    = raw_sp_models[vs]
    print(f'  Hybrid {vs:>5}: {hybrid_path}  ({os.path.getsize(hybrid_path)/1024:.1f} KB)')
    print(f'  Raw    {vs:>5}: {raw_path}  ({os.path.getsize(raw_path)/1024:.1f} KB)')

# 3. List all saved graphs
print('\n✅ Saved graphs:')
for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.png'):
        print(f'  {os.path.join(OUTPUT_DIR, f)}')

print(f'\n✅ All outputs in: {OUTPUT_DIR}')

In [ ]:
# ── Copy everything to Google Drive (optional) ──────────────────────────────
import shutil

# UNCOMMENT BELOW to save to Drive:
# DRIVE_DIR = '/content/drive/MyDrive/npltk_results'
# if os.path.exists(DRIVE_DIR):
#     shutil.rmtree(DRIVE_DIR)
# shutil.copytree(OUTPUT_DIR, DRIVE_DIR)
# print(f'✅ All outputs copied to: {DRIVE_DIR}')

# ── Or download the best model directly ─────────────────────────────────────
# from google.colab import files
# files.download(hybrid_models[16000])
# files.download(csv_path)

print('Done! Uncomment the Drive save or download section above as needed.')

## 📋 Summary

### What was evaluated

| Tokenizer | Description |
|---|---|
| `whitespace` | Naive split on spaces |
| `regex` | Rule-based pre-tokenizer only (Stage 1) |
| `raw_sp_16k` | SentencePiece on raw text (no preprocessing) |
| `hybrid_8k` | Stage 1 + SP Unigram 8k vocab |
| `hybrid_16k` | Stage 1 + SP Unigram 16k vocab |
| `hybrid_32k` | Stage 1 + SP Unigram 32k vocab |

### Key findings
- **OOV rate**: Hybrid ≤ Raw SP (byte_fallback handles unknowns)
- **Tokens/sentence**: Larger vocab = fewer tokens (more efficient)
- **Compression ratio**: Hybrid outperforms whitespace/regex
- **Mixed-script**: Only Hybrid correctly separates Latin from Devanagari

### Next steps
1. Copy `hybrid_16k.model` → `src/npltk/tokenizer/models/nepali_sp.model`
2. Run `pip install -e .` to bundle the model
3. Test with: `from npltk import NepaliHybridTokenizer`